# Case 9: Agentic AI Yapısı

Case 3-8, birbirinden bağımsız çağrılabilen gerçek bileşenler üretti: feature engineering,
4 katmanlı anomali skorlama + birleştirme, configurable rule engine (Case 7), RAG tabanlı politika
açıklaması (Case 8). Bu case, bunları **birden fazla agent'ın birbirine görev devrettiği bir
orkestrasyon katmanına** bağlıyor.

**Mimari:** LangGraph (CrewAI değil); bir referans implementasyondaki
(`dashboard/backend/src/agents`) yerleşik desen birebir izlendi: `state.py` (paylaşılan
`AgentState`), `graph.py` (`StateGraph` + koşullu kenarlar), `shared/llm.py` (`ChatOpenAI` + özel
`base_url`; Ollama'nın OpenAI-uyumlu endpoint'ine işaret ediyor, tek bir env değişkeniyle
OpenRouter/başka bir sağlayıcıya da yönlendirilebilir), her agent kendi klasöründe `agent.py`,
`supervisor/agent.py` giriş noktası (Facade).

**4 agent'tan 3'ü (feature_engineering, anomaly_scoring, rule_engine) tamamen deterministik**:
Case 3/4/5/7'nin zaten doğrulanmış fonksiyonlarını sarmalıyor (Adapter pattern), LLM içermiyor.
Sadece **policy_explanation** gerçek bir LLM node'u (Case 8'in RAG pipeline'ı); bu makinede
Ollama'nın RAM kısıtı yüzünden OpenRouter'ın ücretsiz katmanına geçildi
(`openrouter:nvidia/nemotron-3-embed-1b:free` embedding, `openrouter:nvidia/nemotron-3.5-lightning:free`
LLM), bu adım artık gerçek üretim yapıyor. Sağlayıcı-bağımsız zarif düşüş mekanizması hâlâ kodda
duruyor (Case 8, bölüm 8'de fiilen tetiklenerek doğrulandı); burada tetiklenmiyor çünkü sağlayıcı
gerçekten erişilebilir.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found: expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import json
import pandas as pd

from src.config import settings

pd.set_option("display.max_colwidth", 120)
parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Graf yapısı

`feature_engineering` → `anomaly_scoring` → `rule_engine` → [koşullu] → `policy_explanation` →
END. `anomaly_scoring` ve `rule_engine` HER ZAMAN çalışır (ikisi de ucuz, deterministik);
`policy_explanation`'a geçiş, iki sinyalden (anomali skoru üst-%1'de mi, rule engine verdict'i
HIGH/CRITICAL mi) HERHANGİ biri yüksekse tetiklenir.

In [3]:
from src.agents.graph import build_graph
from src.container import build_container

container = build_container()
rule_engine = container.rule_engine_container.rule_engine()
rag_pipeline = container.rag_container.rag_pipeline()

graph = build_graph(rule_engine, rag_pipeline)
graph_repr = graph.get_graph()

print("Node'lar:")
for node_id in graph_repr.nodes:
    print(f"  - {node_id}")

print()
print("Kenarlar:")
for edge in graph_repr.edges:
    label = f" [{edge.conditional}]" if edge.conditional else ""
    print(f"  {edge.source} -> {edge.target}{label}")


Node'lar:
  - __start__
  - feature_engineering
  - anomaly_scoring
  - rule_engine
  - policy_explanation
  - __end__

Kenarlar:
  __start__ -> feature_engineering
  anomaly_scoring -> rule_engine
  feature_engineering -> anomaly_scoring
  rule_engine -> __end__ [True]
  rule_engine -> policy_explanation [True]
  policy_explanation -> __end__


### Bulunan ve düzeltilen bir tasarım hatası

İlk tasarımda `rule_engine`, sadece anomali skoru yüksekse çalıştırılıyordu (`anomaly_scoring` →
[koşullu] → `rule_engine`). Gerçek bir örnekle doğrulandığında (Case 7'nin fraud_r01 örneği,
TransactionID=2988038) bu işlemin ham anomali skorunun düşük olduğu (0,405, üst-%1 eşiği 0,941'in
çok altında) ama rule engine'de CRITICAL çıktığı görüldü. Case 7'nin kendi notebook'u zaten bunu
göstermişti (bölüm 10: "sadece kurallar" ve "sadece AI" büyük ölçüde farklı işlemleri yakalıyor);
ilk tasarım rule engine'i anomali skoruna bağımlı kılarak, rule engine'in yakaladığı tam da bu tür
işlemleri sistemden dışlıyordu. Düzeltme: `rule_engine` artık her zaman çalışıyor,
`policy_explanation`'a geçiş iki sinyalden herhangi biri yüksekse tetikleniyor; `graph.py`'nin
docstring'inde bu bulgu açıkça not edildi.


## 2. Senaryo 1: düşük riskli işlem (erken çıkış)

In [4]:
from src.agents.supervisor.agent import run_agentic_analysis
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score

all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)
low_risk_id = int(final_raw.loc[final_raw["final_raw_anomaly_score"] < 0.3, "TransactionID"].iloc[0])
print(f"düşük riskli örnek TransactionID: {low_risk_id}")

report_low = run_agentic_analysis(low_risk_id)
print(json.dumps(report_low, indent=2, default=str))

düşük riskli örnek TransactionID: 2987000
{
  "transaction_id": 2987000,
  "risk_level": "low",
  "final_raw_anomaly_score": 0.16154544405802024,
  "rule_verdict": {
    "fired_rules": [],
    "verdict_severity": null,
    "verdict_action": null,
    "verdict_rule_id": null,
    "priority_verdict_rule_id": null
  },
  "policy_explanation": null
}


`rule_engine` çalıştı (hiçbir kural ateşlenmedi: `fired_rules: []`) ama `policy_explanation`
hiç çağrılmadı (`policy_explanation: null`); iki sinyal de düşük olduğu için görev devri burada
duruyor. Bu, delegation mekanizmasının gerçekten çalıştığının kanıtı: pahalı LLM adımı, gerekmediği
zaman atlanıyor.

## 3. Senaryo 2: CRITICAL işlem (tam zincir, agent-to-agent devir)

In [5]:
critical_id = 2988038  # Case 7/8'de kullanılan aynı örnek, fraud_r01'in ateşlendiği işlem

report_critical = run_agentic_analysis(critical_id)
print(json.dumps(report_critical, indent=2, default=str))

{
  "transaction_id": 2988038,
  "risk_level": "low",
  "final_raw_anomaly_score": 0.40517971851209256,
  "rule_verdict": {
    "fired_rules": [
      {
        "rule_id": "fraud_r01",
        "rule_name": "High Amount + Foreign Country + Night",
        "severity": "CRITICAL",
        "action": "BLOCK",
        "priority": 1,
        "condition": "(TransactionAmt=100.0 (gt 70.0) AND addr2=96.0 (is_null False) AND addr2=96.0 (ne 87.0) AND is_low_volume_hour=True (eq True))",
        "message": "Elevated amount (100.0) combined with a foreign billing region (addr2=96.0) during a historically low-volume, high-fraud-rate hour window (04:00-09:00), the exact combination named as an example in the case brief."
      },
      {
        "rule_id": "fraud_r03",
        "rule_name": "New Device + New Address Together",
        "severity": "HIGH",
        "action": "REVIEW",
        "priority": 3,
        "condition": "(is_new_device_for_card=True (eq True) AND is_new_addr1_for_card=True (eq Tru

**Tam zincir çalıştı:** `feature_engineering` → `anomaly_scoring` (risk_level="low", ama skor
state'te taşınıyor) → `rule_engine` (fraud_r01 CRITICAL + fraud_r03 HIGH ateşlendi) → koşullu kenar
rule-sinyalini görüp `policy_explanation`'a devrediyor (anomali skoru düşük olmasına rağmen) →
Case 8'in RAG pipeline'ı otomatik bir soru üretip ilgili policy'leri getiriyor VE gerçek LLM
(`openrouter:nvidia/nemotron-3.5-lightning:free`) cevabı üretiyor; `answer` artık dolu,
`question`/`sources`/`note` ile birlikte.


## 4. Agent-to-agent iletişim: state'in akışı

Agent'lar birbirine doğrudan mesaj göndermiyor; hepsi paylaşılan `AgentState` üzerinden konuşuyor
(Mediator-tarzı iletişim; LangGraph'ın standart, idiomatik yolu). Her node'un girdi/çıktısını
tek tek görelim.

In [6]:
from src.agents.schemas.state import AgentState

initial_state: AgentState = {
    "transaction_id": critical_id, "features": None, "anomaly_scores": None,
    "risk_level": "", "rule_verdict": None, "policy_explanation": None, "report": None,
}

for event in graph.stream(initial_state, {"recursion_limit": 10}):
    node_name = list(event.keys())[0]
    output_keys = list(event[node_name].keys())
    print(f"[{node_name}] state'e yazdığı alanlar: {output_keys}")

[feature_engineering] state'e yazdığı alanlar: ['features']
[anomaly_scoring] state'e yazdığı alanlar: ['anomaly_scores', 'risk_level']
[rule_engine] state'e yazdığı alanlar: ['rule_verdict']


[policy_explanation] state'e yazdığı alanlar: ['policy_explanation']


Her node, `AgentState`'in sadece kendi ürettiği alanları yazıyor (LangGraph bunları birleştirip
sonraki node'a paylaşılan state olarak geçiriyor); `rule_engine` node'u `anomaly_scoring`'in
yazdığı `anomaly_scores`'u OKUYAMAsa bile (kendi state'ini bağımsız yeniden hesaplıyor, bkz.
`rule_engine/agent.py` docstring'i: her agent'ın kendi başına, serileştirilebilir state dışında
hiçbir şey paylaşmaması, dağıtık bir agent sisteminin gerçek sınırını simüle ediyor), `graph.py`'nin
koşullu kenarı hem `risk_level` hem `rule_verdict`'i state'ten okuyup delegasyon kararını veriyor;
agent-to-agent iletişimin somut karşılığı bu.

---

**Durum:** Case 9 (Agentic AI) tamamlandı. LangGraph ile kurulan 4-agent'lı sistem (3'ü
deterministik, 1'i LLM-tabanlı), mevcut bir referans proje deseniyle birebir uyumlu bir
dosya yapısında (`state.py`/`graph.py`/`shared/llm.py`/her agent kendi klasöründe `agent.py`/
`supervisor/agent.py` giriş noktası) inşa edildi. Gerçek bir tasarım hatası (rule engine'in
anomali skoruna yanlış bağımlı kılınması) test sırasında bulundu ve düzeltildi. Task delegation
(iki sinyale göre koşullu geçiş) ve agent-to-agent iletişim (paylaşılan state) hem düşük-riskli
hem CRITICAL gerçek işlemlerle doğrulandı. `policy_explanation`'ın gerçek LLM üretimi de artık
uçtan uca çalışıyor (OpenRouter'ın ücretsiz katmanı, RAM kısıtlı bu makinede Ollama'nın yerini
alıyor; sağlayıcı değişimi `shared/llm.py`'nin `base_url` config'iyle tek satırla mümkün kalmaya
devam ediyor). Bu, projenin son case'i.